# Milestone 5: Statistical Analysis

### Objective

This analysis uses hypothesis testing and 95% confidence intervals to investigate two business questions for Mwangaza MiniMart:

1. Do customers who pay using M-Pesa spend more per order than those using other payment methods?
2. Do promotional orders have a different average value compared with non-promotional orders?

For each question, we will check the assumptions of the statistical test, interpret the results, and explain what they mean for the business.

## 5.1 Import Necessary libraries and Loading datasets

In [1]:
# Import libraries
import pandas as pd
from scipy import stats

In [2]:
# Load cleaned orders dataset
orders = pd.read_csv("../data/processed/orders.csv")

# Display the first five rows
display(orders.head())

# Check the number of rows and columns
print("Dataset shape:", orders.shape)


,order_id,store_id,order_datetime,customer_id,channel,payment_method,order_date,item_units,gross_amount_kes,discount_amount_kes,net_amount_kes,promotion_applied,year,month,year_month
0,O0000001,S002,2024-01-01 08:06:40,C003000,In-store,M-Pesa,2024-01-01,4,1031.00,0.00,1031.00,False,2024,1,2024-01
1,O0000002,S002,2024-01-01 08:08:38,C003410,In-store,M-Pesa,2024-01-01,1,178.38,26.76,151.62,True,2024,1,2024-01
2,O0000003,S004,2024-01-01 08:15:46,C007400,In-store,M-Pesa,2024-01-01,3,1173.67,42.41,1131.26,True,2024,1,2024-01
3,O0000004,S004,2024-01-01 08:19:06,NaN,In-store,M-Pesa,2024-01-01,7,1150.12,0.00,1150.12,False,2024,1,2024-01
4,O0000005,S002,2024-01-01 08:23:39,C003652,In-store,M-Pesa,2024-01-01,5,825.89,0.00,825.89,False,2024,1,2024-01


Dataset shape: (316277, 15)


## 5.2 Exploring Payment Methods

In [3]:
# Count orders by payment method
orders["payment_method"].value_counts(dropna=False)

payment_method
M-Pesa           193915
Cash              72157
Debit card        29269
Credit card       13406
Bank transfer      6739
Unknown             791
Name: count, dtype: int64

### 5.3 Hypothesis Test 1: M-Pesa vs Other Payment Methods

We compare the average order value for M-Pesa transactions
against other known payment methods.

- H0: The two groups have equal average order values.
- H1: The two groups have different average order values.
- Significance level: 0.05 (5%).

We use Welch's t-test and a 95% confidence interval.

In [4]:
# Exclude orders with unknown payment methods
known_orders = orders[orders["payment_method"] != "Unknown"]

# Separate the two groups
mpesa = known_orders.loc[
    known_orders["payment_method"] == "M-Pesa", "net_amount_kes"
].dropna()

other = known_orders.loc[
    known_orders["payment_method"] != "M-Pesa", "net_amount_kes"
].dropna()

# Check group sizes and distributions
print("M-Pesa orders:", len(mpesa))
print("Other payment orders:", len(other))

print("\nM-Pesa mean:", round(mpesa.mean(), 2))
print("Other payments mean:", round(other.mean(), 2))

print("\nM-Pesa standard deviation:", round(mpesa.std(), 2))
print("Other payments standard deviation:", round(other.std(), 2))

print("\nM-Pesa skewness:", round(mpesa.skew(), 2))
print("Other payments skewness:", round(other.skew(), 2))

# Welch's t-test
test = stats.ttest_ind(mpesa, other, equal_var=False)

# 95% confidence interval
ci = test.confidence_interval(confidence_level=0.95)

print("\nMean difference (KES):", round(mpesa.mean() - other.mean(), 2))
print("P-value:", test.pvalue)
print("95% Confidence interval:", ci)

M-Pesa orders: 193915
Other payment orders: 121571

M-Pesa mean: 803.67
Other payments mean: 801.91

M-Pesa standard deviation: 583.33
Other payments standard deviation: 583.11

M-Pesa skewness: 1.35
Other payments skewness: 1.38

Mean difference (KES): 1.76
P-value: 0.40909225981986047
95% Confidence interval: ConfidenceInterval(low=np.float64(-2.4203712608038783), high=np.float64(5.9426918533804125))


### Results and Conclusion: M-Pesa vs Other Payments

M-Pesa orders averaged **KES 803.67**, compared to **KES 801.91** for other payments.

Welch's t-test showed no statistically significant difference (**p = 0.4091**). The 95% confidence interval was **KES -2.42 to 5.94**.

**Assumptions:** Both groups had large samples, although order values were skewed and repeat customers may affect independence.

**Conclusion:** There is insufficient evidence that M-Pesa transactions have a different average value. These findings are based on synthetic data.

In [5]:
# Check promotion values
print(orders["promotion_applied"].value_counts(dropna=False))

# Separate promotional and non-promotional orders
promo = orders.loc[
    orders["promotion_applied"] == True, "net_amount_kes"
].dropna()

no_promo = orders.loc[
    orders["promotion_applied"] == False, "net_amount_kes"
].dropna()

# Descriptive statistics and assumption checks
print("\nPromotional orders:", len(promo))
print("Non-promotional orders:", len(no_promo))

print("\nPromotional mean:", round(promo.mean(), 2))
print("Non-promotional mean:", round(no_promo.mean(), 2))

print("\nPromotional standard deviation:", round(promo.std(), 2))
print("Non-promotional standard deviation:", round(no_promo.std(), 2))

print("\nPromotional skewness:", round(promo.skew(), 2))
print("Non-promotional skewness:", round(no_promo.skew(), 2))

# Welch's t-test
promo_test = stats.ttest_ind(promo, no_promo, equal_var=False)

# 95% confidence interval
promo_ci = promo_test.confidence_interval(confidence_level=0.95)

print("\nMean difference (KES):", round(promo.mean() - no_promo.mean(), 2))
print("P-value:", promo_test.pvalue)
print("95% Confidence interval:", promo_ci)

promotion_applied
False    230764
True      85513
Name: count, dtype: int64

Promotional orders: 85513
Non-promotional orders: 230764

Promotional mean: 941.43
Non-promotional mean: 751.71

Promotional standard deviation: 609.4
Non-promotional standard deviation: 564.68

Promotional skewness: 1.17
Non-promotional skewness: 1.46

Mean difference (KES): 189.72
P-value: 0.0
95% Confidence interval: ConfidenceInterval(low=np.float64(185.02992750222657), high=np.float64(194.40894938868004))


### Results and Conclusion: Promotional vs Non-Promotional Orders

Promotional orders averaged **KES 941.43**, compared to **KES 751.71** for non-promotional orders.

Welch's t-test showed a statistically significant difference (**p < 0.001**). The 95% confidence interval for the difference was **KES 185.03 to 194.41**.

**Assumptions:** Both groups had large samples. Order values were skewed, and repeat customers may affect independence.

**Conclusion:** Promotional orders were associated with higher spending. However, this does not prove promotions caused the increase. The findings use synthetic data.